In [2]:
zcrit, uncrit, SNRpix, SNRbin, combpost = 0.2, 0.5, 2, 5, 1e-12

In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
import warnings
from astropy.visualization import ZScaleInterval
import os
import h5py
from numba import jit
import time
from astropy.table import Table
import seaborn as sns
import matplotlib.colors as mcolors
import glob


In [ ]:
# import zmap, SNRmap
f = h5py.File('/Users/ncv375/Desktop/Year2/lazy/Lazy.jl/mosaic.h5', 'r')
zmap = f['photoz/z_values'][:]
SNRmap = f['SNRmap/SNR_F444W_values'][:]

f444 = fits.open('/Volumes/dataSSD/GOODSS/goodss_nrc_f444w_sci.fits')[0].data

photcat = Table.read('/Users/ncv375/Desktop/Year2/' + 'natalia_selcols_goodss_photom_v0.891_massivegalaxies_22326.fits', format='fits', hdu=1)

intz = fits.open('/Volumes/dataSSD/GOODSS/goodss_intzmap_v0.95.fits')[0].data

In [ ]:
# function to create cutouts
def cutouts(ID, RA, Dec, bbox, zmap, SNRmap, chunks, f444, intz, padding=20):
    '''
    This function makes cutouts of the pixel redshift mosaic and chunked P(z) mosaicto perform pixel binning on per source (or per grouped sources).

    Input:
    - pixel redshift mosaic
    - P(z) chunked mosaics
    - catalog with selected columns only (need central pixel and bounding box information per source)

    Output:
    - cutout of pixel redshift mosaic per source
    - cutout of P(z) chunked mosaics per source
    '''
    
    # set out the variables from inputs
    print('Grab cutout information from catalog...')
    xmin, xmax, ymin, ymax = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
    xminpad, xmaxpad = max(0, xmin - padding), min(zmap.shape[1], xmax + padding)
    yminpad, ymaxpad = max(0, ymin - padding), min(zmap.shape[0], ymax + padding)

    print('Initializing cutouts...')
    # initialize cutouts
    cutoutz = np.full((2, ymaxpad - yminpad, xmaxpad - xminpad), np.nan)
    cutoutPz = np.full((1001, ymaxpad - yminpad, xmaxpad - xminpad), np.nan) # @todo generalize Pz dimensions

    # make the cutout for the redshift mosaic
    cutoutz[0] = zmap[yminpad:ymaxpad, xminpad:xmaxpad]
    cutoutz[1] = SNRmap[yminpad:ymaxpad, xminpad:xmaxpad]
    f444cutout = f444[yminpad:ymaxpad, xminpad:xmaxpad]
    intzcutout = intz[yminpad:ymaxpad, xminpad:xmaxpad]

    print('Beginning to fill cutout for P(z) mosaic...')
    for i in range(len(chunks)):
        with h5py.File('/Volumes/dataSSD/lazymosaic_zcats/Pz_chunks/lazymosaic_sec'+str(chunks[i])+'_Pz.hdf5', 'r') as f: # @todo need to create these files for v0.95 mosaic and outputs
            Pzchunk = f['Pz/Pz_values'][:]
            coordschunk = f['coordinates/pixel_coordinates'][:]

            # diagnostic print statements
            print(f'  Pzchunk shape: {Pzchunk.shape}')
            print(f'  coordschunk shape: {coordschunk.shape}')
            print(f'  Non-NaN in Pzchunk: {np.sum(~np.isnan(Pzchunk))}') # @todo delete these print statements for automated runs once validated

            x = coordschunk[:, 0] + (372*chunks[i]) # @todo generalize this dimension for other mosaics
            y = coordschunk[:, 1]

            # diagnostic print statements
            print(f'  x range: {x.min()} to {x.max()}')
            print(f'  y range: {y.min()} to {y.max()}')
            print(f'  Looking for x in [{xminpad}, {xmaxpad}), y in [{yminpad}, {ymaxpad})') # @todo delete these print statements for automated runs once validated

            mask = (
                (x >= xminpad) & (x < xmaxpad) &
                (y >= yminpad) & (y < ymaxpad))

            if not np.any(mask):
                # no pixels in range for this chunk, skipping
                continue # @todo can restructure this later to evaluate earlier in process, without having to open every output file

            xmap = x[mask] - xminpad
            ymap = y[mask] - yminpad

            cutoutPz[:, ymap, xmap] = Pzchunk[mask].T

    # diagnostic print statement
    print('shapes:', cutoutz.shape, cutoutPz.shape, f444cutout.shape)    

    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20, 5))

    im1 = ax1.imshow(cutoutz[0], cmap='jet', origin='lower')
    fig.colorbar(im1, ax=ax1, label=r'photo z', location='right')
    ax1.set_title('pixel z map')

    im2 = ax2.imshow(np.nansum(cutoutPz, axis=0), cmap='jet', origin='lower')
    fig.colorbar(im2, ax=ax2, label=r'Pz sum', location='right')
    ax2.set_title('sum(Pz)')

    im3 = ax3.imshow(cutoutz[1], cmap='jet', origin='lower')
    fig.colorbar(im3, ax=ax3, label=r'SNR', location='right')
    ax3.set_title('F444W SNR map')

    interval = ZScaleInterval()
    vmin, vmax = interval.get_limits(f444cutout)

    im4 = ax4.imshow(f444cutout, cmap='gray_r', vmin=vmin, vmax=vmax, origin='lower')
    fig.colorbar(im4, ax=ax4, label=r'F444W flux', location='right')
    ax4.set_title('F444W map')

    plt.suptitle('ID = ' + str(ID) + ',\nRA, Dec = ' + str(RA) + ', ' + str(Dec))
    plt.savefig(fname='/Users/ncv375/Desktop/Year2/massivegal_cutouts/'+str(ID)+'cutouts.png', dpi=350, bbox_inches='tight')
    plt.show()

    return cutoutz, cutoutPz, intzcutout

In [ ]:
@jit(nopython=False, error_model='numpy', parallel=False)
def combine_pdfs(postgal, binmask):
    # initialize array to store Pz product
    product = np.ones(1001) # lenPz = 1001 # @todo generalize Pz dimensions
    binspacing = 0.02 # @todo generalize binspacing for other Lazy configurations

    masked_pdfarr = np.where(binmask[None, :, :], postgal, 1.0) # sets up array with same shape as postgal, where all pixels in binmask have postgal value, and other pixels have 1.0 (equivalent to 0 probability in log space)
    logsum = np.sum(np.sum(np.log10(masked_pdfarr), axis=1), axis=1) # perform operations on log space of PDF that are equivalent to multiplying the PDF (two np.sum operations to collapse over the y then x dimensions)
    product = 10**(logsum - max(logsum)) # project combined PDF back to linear space, subtract max for numerical stability
    pdf_combined = product/(np.sum(product)*binspacing) # normalize the combined PDF to ensure it integrates to 1

    return pdf_combined